In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import pickle

from mci_progression_ml.dataset.dataset import create_dataset
from mci_progression_ml.framework.nested_cv.method import nested_cv
from mci_progression_ml.framework.nested_cv.dataset_split import create_train_test_split
from mci_progression_ml.framework.model_config import xgb, rf, lr, svc
from mci_progression_ml.framework.nested_cv.summary import summarize_family
from mci_progression_ml.utils import convert
from mci_progression_ml.config import RANDOM_STATE, cog, dem, fs_col, num_features, cat_features
from mci_progression_ml.framework.final.method import final_deployment_model
from mci_progression_ml.framework.final.metrics import bootstrap_metrics
from mci_progression_ml.framework.utils import inspect_dropped_features, get_selected_feature_names

In [2]:
pd.set_option('display.float_format', '{:.3f}'.format)

In [3]:
df = create_dataset()

DIAGNOSIS_24
0    1306
1     702
Name: count, dtype: int64


In [4]:
outer_cv, \
cv, \
X_train, \
X_test, \
y_train, \
y_test, \
groups_train, \
groups_test, \
diag_change_train, \
diag_change_test = create_train_test_split(
    df,
    ocv_n_splits=5,
    ocv_n_repeats=5,
    icv_n_splits=5,
)


In [ ]:
# run all 84 configurations (4 models x 7 modalities x 3 sampling methods)
results = []
for model in [xgb, rf, lr, svc]:
    for modality in ["dem", "cog", "mri", "dem_cog", "cog_mri", "mri_dem", "dem_cog_mri"]:
        for sampling in ["none", "smote", "smote_tomek"]:
            result = nested_cv(X_train, y_train, groups_train, cv, outer_cv, model, sampling, modality, diag_change_train, xai=False)
            results.append({f"{model.__class__.__name__.lower()}-{modality}-{sampling}": result})


Path("results").mkdir(parents=True, exist_ok=True)

with open("results/nested_cv_results.json", "w") as f:
    json.dump(results, f, indent=4, default=convert)

In [4]:
with open('results/nested_cv_results.json', 'r') as file:
    results = json.load(file)

In [10]:
# ---- Build summary dataframe for Logistic Regression variants ----
summary = [
    summarize_family(k, v)
    for result in results
    for k, v in result.items()
]

df = pd.DataFrame(summary)

# Split the family column into parts
# Format: model-modality-sampling
df[['model', 'modality', 'sampling']] = df['family'].str.split('-', expand=True)
df['sampling'] = df['sampling'].replace({
    'smote': 'SMOTENC',
    'smote_tomek': 'SMOTENC-Tomek',
    'none': 'None'
})
df['model'] = df['model'].str.upper().str.replace('CLASSIFIER', '', regex=False).str.strip()
df['model'] = df['model'].replace({
    'LOGISTICREGRESSION': 'LR',
    'RANDOMFOREST': 'RF',
})

df['modality'] = df['modality'].str.upper().str.replace('_', '-').str.strip()


# Sort by macro F1 mean (primary), then PR-AUC mean (secondary)
df = df.sort_values(
    by="balanced_acc_mean",
    ascending=False
).reset_index(drop=True)

In [ ]:
# Top 5 configurations based on balanced accuracy mean
df[['model', 'modality', 'sampling', 'f1_macro_mean', 'f1_macro_sd', 'balanced_acc_mean', 'balanced_acc_sd',
    'pr_auc_mean', 'pr_auc_sd', 'sensitivity_mean', 'sensitivity_sd', 'specificity_mean', 'specificity_sd'
    ]].sort_values(
    by="balanced_acc_mean",
    ascending=False
).head(5)

,model,modality,sampling,f1_macro_mean,f1_macro_sd,balanced_acc_mean,balanced_acc_sd,pr_auc_mean,pr_auc_sd,sensitivity_mean,sensitivity_sd,specificity_mean,specificity_sd
0,XGB,DEM-COG-MRI,SMOTENC,0.797,0.029,81.278,2.563,0.791,0.045,83.168,5.015,79.389,5.537
1,XGB,DEM-COG-MRI,SMOTENC-Tomek,0.794,0.025,81.100,2.322,0.784,0.046,83.313,5.357,78.886,4.801
2,SVC,DEM-COG-MRI,SMOTENC-Tomek,0.792,0.030,80.982,2.919,0.802,0.044,83.411,5.301,78.553,4.491
3,SVC,DEM-COG-MRI,SMOTENC,0.790,0.031,80.913,2.892,0.799,0.046,83.985,5.204,77.841,5.021
4,XGB,DEM-COG-MRI,None,0.788,0.029,80.805,2.954,0.790,0.046,84.304,5.573,77.306,4.297


In [ ]:
# Run the best model (XGB) with the best modality (DEM-COG-MRI) and best sampling method (SMOTENC) with XAI enabled
result = nested_cv(X_train, y_train, groups_train, cv, outer_cv, xgb, "smote", "dem_cog_mri", diag_change_train, xai=True)


with open("results/nested_cv_result_best_configuration_with_xai.json", "w") as f:
    # json.dump(results, f, indent=4, default=convert)
    json.dump(result, f, indent=4, default=convert)

In [6]:
#final deployable model

Path("results").mkdir(parents=True, exist_ok=True)

all_features = dem + cog + fs_col
test_set_run = [
    # (all_features, "All Features"),
    (fs_col, "Functional Connectivity"),
    (cog, "Cognitive Scores"),
    (dem, "Demographics"),
    ([f for f in all_features if f != "APOE4"], "Without APOE4"),
    ([f for f in all_features if f != "MMSCORE"], "Without MMScore"),
    ([f for f in all_features if f != "CDRSB"], "Without CDRSB"),
    ([f for f in all_features if f != "TOTAL13"], "Without TOTAL13"),
    ([f for f in all_features if f != "FAQTOTAL"], "Without FAQTOTAL")
]

for run in test_set_run:
    features, name = run
    best_model, final_model, best_threshold, best_params = final_deployment_model(X_train, y_train, groups_train, xgb, "smote",
                                                                features, name, n_splits=5, n_repeats=5)

    Path("models").mkdir(parents=True, exist_ok=True)
    with open(f"models/final_model_{name.replace(' ', '_').lower()}.pkl", "wb") as f:
            pickle.dump(final_model, f)

    with open(f"models/best_model_{name.replace(' ', '_').lower()}.pkl", "wb") as f:
        pickle.dump(best_model, f)

    result = bootstrap_metrics(final_model, X_test, y_test, n_bootstrap=2000, random_state=42)
    dropped = inspect_dropped_features(best_model, num_features)

    result.update({
        "selected_features": get_selected_feature_names(best_model),
        "variance_dropped_features": dropped["variance_dropped"],
        "correlation_dropped_features": dropped["correlation_dropped"],
        "params": best_params,
    })

    with open(f"results/final_model_independent_test_set_{name.replace(' ', '_').lower()}.json", "w") as f:
        # json.dump(results, f, indent=4, default=convert)
        json.dump(result, f, indent=4, default=convert)

Running Functional Connectivity
Fitting 25 folds for each of 1000 candidates, totalling 25000 fits
Running Cognitive Scores
Fitting 25 folds for each of 1000 candidates, totalling 25000 fits
Running Demographics
Fitting 25 folds for each of 1000 candidates, totalling 25000 fits
Running Without APOE4
Fitting 25 folds for each of 1000 candidates, totalling 25000 fits
Running Without MMScore
Fitting 25 folds for each of 1000 candidates, totalling 25000 fits
Running Without CDRSB
Fitting 25 folds for each of 1000 candidates, totalling 25000 fits
Running Without TOTAL13
Fitting 25 folds for each of 1000 candidates, totalling 25000 fits
Running Without FAQTOTAL
Fitting 25 folds for each of 1000 candidates, totalling 25000 fits
